# Kimee: flip TTA of `x101_2048` (third ensemble member, no training)

Same as Mirul's M2 notebook (`colab_phase6_mirul.ipynb`), but for one model: Kimee's Phase 4 checkpoint
`x101_2048` (exp008, member of the LB 0.38 ensemble). It predicts every val + test image mirrored left-right (`h`),
upside-down (`v`) and both (`hv`). The un-flipped predictions already exist (`results/phase4/`).
On the laptop the 4 views are fused, then fused with Mirul's m28 + r50 TTA files (exp101) -> 3-model ensemble.

| Step | What | Time (estimate) |
|---|---|---|
| 0 | probe: 7 images, one flip | ~5 min after install |
| 1 | 3 flips x 321 images (141 val + 180 test) | T4 ~30-50 min; A100 ~10 min |

**Writes only to** `solar_filament/Phase 6/Kimee/` (checkpoint is only read).
**How to run:** Runtime -> Change runtime type -> T4 (or A100), then Run all. If it disconnects, run all again:
finished files are skipped.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Mount Google Drive (Kimee's own `solar_filament` folder)


In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/solar_filament"
MODELS = {   # name -> flags it was trained with (must match for prediction)
    "x101_2048": "--backbone x101 --resolution 2048",
}
CKPT = {n: f"{DRIVE_DIR}/output/detectron2/{n}/model_final.pth" for n in MODELS}
for n, c in CKPT.items():
    assert os.path.exists(c), f"Checkpoint not found: {c}"
MY_DIR = f"{DRIVE_DIR}/Phase 6/Kimee"        # the ONLY Drive place this notebook writes to
os.makedirs(MY_DIR, exist_ok=True)
OUT_DIR = "/content/out"                     # local scratch; files are copied to MY_DIR
os.environ["SOLAR_OUTPUT_DIR"] = OUT_DIR
print("Reading", CKPT, "| writing to:", MY_DIR)

## 2. Get the latest code (pulls if the repo is already cloned)

In [ ]:
BRANCH = "Dev"
REPO = "github.com/amiruliman592/Kaggle-Competition-.git"
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token}@{REPO}" if token else f"https://{REPO}"

if os.path.exists("/content/solar"):
    !git -C /content/solar pull -q origin {BRANCH}
else:
    !git clone -q --depth 1 -b {BRANCH} {url} /content/solar
%cd /content/solar
os.environ["SOLAR_ROOT"] = "/content/solar"
!git log --oneline -1

## 3. Install dependencies (Detectron2 build ~5–8 min)

In [ ]:
!pip install -q -r requirements.txt
!pip install -q --no-build-isolation 'git+https://github.com/facebookresearch/detectron2.git'
import torch, detectron2
print("torch", torch.__version__, "| detectron2", detectron2.__version__, "| GPU", torch.cuda.get_device_name(0))

## 4. Quick checks (~30 s) and helpers

In [ ]:
import shutil, subprocess, time

def sh(cmd):
    """Run one command (output shown live); stop the notebook if it fails."""
    t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")
    print(f"--- done in {(time.time() - t0) / 60:.1f} min")

def dump_flip(name, flip, probe=False):
    """Predict val + test with one flip; copy the two files to MY_DIR. Skips files that already exist."""
    tag = f"{name}_flip{flip}"
    if not probe and all(os.path.exists(f"{MY_DIR}/raw_preds_{tag}_{s}.json.gz") for s in ("val", "test")):
        print(f"{tag}: already done, skipped.")
        return
    run_name = f"probe_{name}" if probe else name
    sh(f"python run_pipeline.py --step dump --name {run_name} {MODELS[name]} --weights '{CKPT[name]}' --flip {flip}"
       + (" --dry-run" if probe else ""))
    if not probe:
        for s in ("val", "test"):
            shutil.copy(f"{OUT_DIR}/raw_preds_{tag}_{s}.json.gz", MY_DIR)
        print(f"{tag}: saved to {MY_DIR}")

sh("python -m pytest -q -p no:warnings tests/test_pipeline_flags.py tests/test_flip.py tests/test_official_metric.py")

## Step 0: probe (7 images, one flip)
Check `Peak GPU memory` (T4: must stay below ~14 GB) and `s/img`. On CUDA out of memory: stop and tell Claude.

In [ ]:
dump_flip("x101_2048", "h", probe=True)

## Step 1: all 3 flips

In [ ]:
for flip in ("h", "v", "hv"):
    dump_flip("x101_2048", flip)
print(os.listdir(MY_DIR))   # must list 6 raw_preds_x101_2048_flip* files
print("All flips done.")

## 5. Send results back
Download the 6 files into the repo folder `results/phase6/Kimee/` on the laptop, then tell Claude (also copy the
`Peak GPU memory` and `s/img` lines). Then Runtime -> Disconnect and delete runtime.

In [ ]:
from google.colab import files
for flip in ("h", "v", "hv"):
    for s in ("val", "test"):
        f = f"{MY_DIR}/raw_preds_x101_2048_flip{flip}_{s}.json.gz"
        if os.path.exists(f):
            files.download(f)
        else:
            print("not found (not finished?):", f)